In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torchaudio
import IPython.display as ipd
from tqdm import tqdm
assert torch.cuda.is_available()
%cd /mlx/users/zongyu.yin/playground/samantha

sample_rate = 24000
hop_length = 240
shuffle_buffer_size = 10
num_workers = 2
min_duration=2
max_duration=30
batch_size = 2

In [ ]:
import librosa
import matplotlib.pyplot as plt
from librosa.feature.inverse import mel_to_audio
from torchaudio.functional import DB_to_amplitude

def plot_waveform(waveform, sr, title="Waveform", ax=None):
    waveform = waveform.numpy()

    num_channels, num_frames = waveform.shape
    time_axis = torch.arange(0, num_frames) / sr

    if ax is None:
        _, ax = plt.subplots(num_channels, 1)
    ax.plot(time_axis, waveform[0], linewidth=1)
    ax.grid(True)
    ax.set_xlim([0, time_axis[-1]])
    ax.set_title(title)


def plot_spectrogram(specgram, title=None, ylabel="freq_bin", ax=None):
    if ax is None:
        _, ax = plt.subplots(1, 1)
    if title is not None:
        ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.imshow(specgram, origin="lower", aspect="auto", interpolation="nearest")

def mel2audio(mel):
    mel_db2amp = DB_to_amplitude(mel, ref=1, power=1)
    audio = mel_to_audio(mel_db2amp.numpy(), sr=sample_rate, n_fft=512, win_length=400, hop_length=hop_length, n_iter=100)
    return audio
    

# Dataloader

In [ ]:
from recipes.datasets.mcc.mix import MixWebDataModule

pl_datamodule = MixWebDataModule(
    sample_rate=sample_rate,
    batch_size=batch_size * max_duration * sample_rate,
    shuffle_buffer_size=shuffle_buffer_size,
    num_workers=num_workers,
    region="CN",
    weights=[1, 1, 1],
    max_num_crops=None,
    min_duration=min_duration,
    max_duration=max_duration,
    normalize_audio=False,
)

val_loaders = pl_datamodule.val_dataloader()
val_karaoke, val_libritts = val_loaders[0], val_loaders[1]

In [ ]:
val_iter_karaoke, val_iter_libritts = iter(val_karaoke), iter(val_libritts)

In [ ]:
batch_karaoke, batch_libritts = next(val_iter_karaoke), next(val_iter_libritts)

In [ ]:
print("=======Karaoke=======")
for text, audio in zip(batch_karaoke["text"], batch_karaoke["audio"]):
    print(text)
    ipd.display(ipd.Audio(audio, rate=sample_rate))
print("=======LibriTTS=======")
for text, audio in zip(batch_libritts["text"], batch_libritts["audio"]):
    print(text)
    ipd.display(ipd.Audio(audio, rate=sample_rate))

# Model loading

In [ ]:
import os
import samantha.utils.hdfs_helper as hh
from recipes.umm.modules.lit_module import Stage3
from transformers import BertTokenizer

ckpt_path = ".module_cache/umm/umm_chroma_25hz_vq32768x32_warmup30000_baseline/checkpoints/step=030000.ckpt"
# ckpt_path = ".module_cache/umm/umm_chroma_25hz_vq32768x32_warmup30000_mcc1m/checkpoints/step=030000.ckpt"
umm_model = Stage3.load_from_checkpoint(ckpt_path).to("cuda").eval()
umm_model.tokenizer = BertTokenizer.from_pretrained('bert-large-uncased')

## Evaluation tools

In [ ]:
class GreedyCTCDecoder(torch.nn.Module):
    def __init__(self, labels, blank=0):
        super().__init__()
        self.labels = labels
        self.blank = blank

    def forward(self, emission_batch):
        """Given a sequence emission over labels, get the best path
        Args:
          emission_batch (Tensor): Logit tensors. Shape `[batch, num_seq, num_label]`.

        Returns:
          List[str]: The resulting transcript
        """
        res = []
        for emission in emission_batch:
            indices = torch.argmax(emission, dim=-1)  # [num_seq,]
            indices = torch.unique_consecutive(indices, dim=-1)
            indices = [i for i in indices if i != self.blank]
            joined = " ".join([self.labels[i] for i in indices])
            res.append(joined.replace("|", " ").strip())
        return res

vocab = list(umm_model.tokenizer.get_vocab().keys())
greedy_decoder = GreedyCTCDecoder(vocab)


@torch.no_grad()
def wer(it, verbose=True, max_i=1000):
    mean_wer = []
    mean_mel_loss = []
    mean_chroma_loss = []
    for i, batch in tqdm(enumerate(it)):
        input_batch = {"audio": batch["audio"].to("cuda"), "text": batch["text"]}
        model_input = umm_model.prepare_feature(input_batch)
        model_output = umm_model.model(model_input)

        loss_dict = umm_model.criterion(
            recon_feature=model_output["recon_feature"],
            feature=model_input["feature"],
            logits=model_output["logits"],
            text_ids=model_input["text_ids"],
            recon_chroma=model_output["recon_chroma"],
            chroma=model_input["chroma"],
        )
        mean_mel_loss.append(loss_dict["stft_loss"])
        mean_chroma_loss.append(loss_dict["chroma_stft_loss"])
        
        actual_transcript = batch["text"]
        greedy_transcript = greedy_decoder(model_output["logits"])
        for j, (a, g) in enumerate(zip(actual_transcript, greedy_transcript)):
            greedy_wer = torchaudio.functional.edit_distance(a, g) / len(a)
            if verbose:
                print("=============================")
                print(f"Actual transcript: {a}")
                print(f"Greedy transcript: {g}")
                print(f"WER: {greedy_wer}")
                ipd.display(ipd.Audio(batch["audio"][j], rate=sample_rate))
            mean_wer.append(greedy_wer)
        if max_i is not None and i >= max_i:
            break
    mean_mel_loss = sum(mean_mel_loss) / len(mean_mel_loss)
    mean_chroma_loss = sum(mean_chroma_loss) / len(mean_chroma_loss)
    mean_wer = sum(mean_wer) / len(mean_wer)
    print(f"Mean Mel loss: {mean_mel_loss}")
    print(f"Mean Chroma loss: {mean_chroma_loss}")
    print(f"Mean WER: {mean_wer}")
    return mean_mel_loss, mean_chroma_loss, mean_wer

## Greedy

In [ ]:
from recipes.datasets.mcc.mix import MixWebDataModule

pl_datamodule = MixWebDataModule(
    sample_rate=sample_rate,
    batch_size=batch_size * max_duration * sample_rate,
    shuffle_buffer_size=shuffle_buffer_size,
    num_workers=num_workers,
    region="CN",
    weights=[1, 1, 1],
    max_num_crops=None,
    min_duration=min_duration,
    max_duration=max_duration,
    normalize_audio=False,
)

val_loaders = pl_datamodule.val_dataloader()
val_karaoke, val_libritts = val_loaders[0], val_loaders[1]
val_iter_karaoke, val_iter_libritts = iter(val_karaoke), iter(val_libritts)

In [ ]:
max_i = None
verbose = False
eval_out = wer(val_iter_karaoke, verbose, max_i)

In [ ]:
max_i = None
verbose = False
eval_out = wer(val_iter_libritts, verbose, max_i)

# VQ Tokens

In [ ]:
from tqdm import tqdm
from recipes.datasets.mcc.mix import VocalWebDataModule

pl_datamodule = VocalWebDataModule(
    sample_rate=sample_rate,
    batch_size=batch_size,
    shuffle_buffer_size=shuffle_buffer_size,
    num_workers=num_workers,
)

val_loaders = pl_datamodule.val_dataloader()
vocal_loader, speech_loader = val_loaders[0], val_loaders[1]
vocal_iter, speech_iter = iter(vocal_loader), iter(speech_loader)

def w2t(it, verbose=False, max_i=1000):
    tokens = []
    for i, batch in tqdm(enumerate(it)):
        wav = batch["audio"].to("cuda")
        vq_ids = umm_model.wav2token(wav)
        for j, (w, t) in enumerate(zip(wav, vq_ids)):
            if verbose:
                print("=============================")
                print(t)
                ipd.display(ipd.Audio(w.cpu(), rate=sample_rate))
            tokens.append(t.unique().cpu())
        if i >= max_i:
            break
    print(len(torch.cat(tokens, dim=0).unique()))

In [ ]:
max_i = 10000
verbose = False
w2t(vocal_iter, verbose, max_i)

In [ ]:
max_i = 10000
verbose = False
w2t(speech_iter, verbose, max_i)